In [40]:
import numpy as np

In [41]:
import pandas as pd

In [42]:
dataset = pd.read_csv("Social_Network_Ads.csv")

In [43]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [44]:
dataset = pd.get_dummies(dataset, dtype =int, drop_first=True)

In [45]:
dataset.drop("User ID", axis=1)

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [46]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [47]:
independent = dataset[["Age", "EstimatedSalary", "Gender_Male"]]

In [48]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [49]:
dependent = dataset[["Purchased"]]

In [50]:
dependent

,Purchased
0,0
1,0
2,0
3,0
4,0
...,...
395,1
396,1
397,1
398,0


In [51]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(independent, dependent, test_size =0.30, random_state=0)

In [52]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [53]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
param_grid = {'criterion':['gini','entropy'],'max_features':['sqrt','log2'],'splitter':['best','random']}
grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit=True, verbose = 3, n_jobs=-1, scoring = 'f1_weighted')
grid.fit(x_train, y_train)
re=grid.cv_results_
grid_predictions = grid.predict(x_test)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


In [54]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

In [55]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [56]:
#from sklearn.metrics import f1_score
#f1_macro=f1_score(y_test, grid_predictions, average = 'weighted')
#print("The f1_macro value for best parameter{}:".format(grid.best_params_),f1_macro)
print("The best parameter set for this model:\n", format(grid.best_params_))
print("The Confusion Matrix:\n",cm)
print("The report:\n", clf_report)
from sklearn.metrics import roc_auc_score
ROC_score = roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])
print("The ROC_AUC score for this model is:\n", ROC_score )


The best parameter set for this model:
 {'criterion': 'entropy', 'max_features': 'log2', 'splitter': 'best'}
The Confusion Matrix:
 [[72  7]
 [ 5 36]]
The report:
               precision    recall  f1-score   support

           0       0.94      0.91      0.92        79
           1       0.84      0.88      0.86        41

    accuracy                           0.90       120
   macro avg       0.89      0.89      0.89       120
weighted avg       0.90      0.90      0.90       120

The ROC_AUC score for this model is:
 0.8947205927755479


In [57]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.005818,0.000634,0.015201,0.001709,gini,sqrt,best,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.855314,0.804584,0.823129,0.822861,0.891667,0.839511,0.030775,3
1,0.005650,0.001274,0.015133,0.001967,gini,sqrt,random,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.875644,0.874254,0.750000,0.816856,0.853485,0.834048,0.047083,4
2,0.004490,0.000368,0.016016,0.002104,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.892857,0.823129,0.787755,0.804432,0.800053,0.821645,0.037375,5
3,0.004256,0.000407,0.014224,0.001132,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.785714,0.838326,0.772092,0.830519,0.874356,0.820202,0.037072,6
4,0.005184,0.000711,0.013114,0.001051,entropy,sqrt,best,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.855314,0.799537,0.841398,0.823781,0.926743,0.849355,0.042950,2
5,0.004111,0.000642,0.011830,0.000850,entropy,sqrt,random,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.791441,0.785714,0.701966,0.839990,0.802559,0.784334,0.045314,8
6,0.003837,0.000392,0.012025,0.000618,entropy,log2,best,"{'criterion': 'entropy', 'max_features': 'log2...",0.855314,0.893878,0.789152,0.892857,0.946153,0.875471,0.051950,1
7,0.003823,0.000721,0.011128,0.001200,entropy,log2,random,"{'criterion': 'entropy', 'max_features': 'log2...",0.819142,0.802399,0.752381,0.838458,0.787975,0.800071,0.029200,7


In [58]:
Age_input = float(input("Age="))
Est_salary_input = float(input("Estimated Salary="))
Gender_male_input = int(input("Gender Male="))

Age= 43
Estimated Salary= 7000
Gender Male= 0


In [59]:
Future_predictions = grid.predict([[Age_input, Est_salary_input, Gender_male_input]])
print("Future Predictions:\n", Future_predictions)

Future Predictions:
 [1]
